# Week 8 - Task 3: LoRA from Scratch

This notebook implements `LoRALinear` in raw PyTorch, verifies the required correctness properties, and answers the conceptual questions from `DELIVERABLES.md`.

In [1]:
from pathlib import Path
import sys

import numpy as np
import torch
import torch.nn as nn

repo_root = Path.cwd().resolve()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from src.models import LoRALinear


In [2]:
SEED = 42
np.random.seed(SEED)
_ = torch.manual_seed(SEED)

## Task 3.1 - Implement `LoRALinear`

The reusable implementation lives in `src/models.py` and is imported into this notebook. The wrapped base linear layer is frozen, while the low-rank matrices `A` and `B` remain trainable. This notebook uses the convention `A = (input_dim, rank)` and `B = (rank, output_dim)`.

In [3]:
print("Implementation imported from src.models.LoRALinear")
print("LoRA shape convention: A = (input_dim, rank), B = (rank, output_dim)")

Implementation imported from src.models.LoRALinear
LoRA shape convention: A = (input_dim, rank), B = (rank, output_dim)


## Task 3.2 - Verify Correctness

Each verification group below prints its result explicitly and uses assertions so a failure stops notebook execution immediately.

In [4]:
forward_linear = nn.Linear(768, 768)
forward_lora = LoRALinear(forward_linear, rank=8, alpha=16)
forward_x = torch.randn(4, 32, 768)
forward_output = forward_lora(forward_x)

has_nan = bool(torch.isnan(forward_output).any().item())
has_inf = bool(torch.isinf(forward_output).any().item())

assert forward_output.shape == (4, 32, 768)
assert not has_nan
assert not has_inf

print("Forward pass verification: PASS")
print(f"output shape: {tuple(forward_output.shape)}")
print(f"contains NaN: {has_nan}")
print(f"contains Inf: {has_inf}")

Forward pass verification: PASS
output shape: (4, 32, 768)
contains NaN: False
contains Inf: False


In [5]:
grad_linear = nn.Linear(768, 768)
grad_lora = LoRALinear(grad_linear, rank=8, alpha=16)
grad_x = torch.randn(4, 32, 768)
grad_loss = grad_lora(grad_x).pow(2).mean()
grad_loss.backward()

a_grad_present = grad_lora.A.grad is not None
b_grad_present = grad_lora.B.grad is not None
weight_grad_present = grad_lora.linear.weight.grad is not None
bias_grad_present = (
    grad_lora.linear.bias.grad is not None if grad_lora.linear.bias is not None else False
)

assert a_grad_present
assert b_grad_present
assert not weight_grad_present
assert not bias_grad_present

print("Gradient isolation verification: PASS")
print(f"A.grad present: {a_grad_present}")
print(f"B.grad present: {b_grad_present}")
print(f"base weight grad present: {weight_grad_present}")
print(f"base bias grad present: {bias_grad_present}")

Gradient isolation verification: PASS
A.grad present: True
B.grad present: True
base weight grad present: False
base bias grad present: False


In [6]:
zero_linear = nn.Linear(768, 768)
zero_lora = LoRALinear(zero_linear, rank=8, alpha=16)
zero_x = torch.randn(4, 32, 768)
base_output = zero_lora.linear(zero_x)
wrapped_output = zero_lora(zero_x)
matches_base = torch.allclose(wrapped_output, base_output)

assert matches_base

print("Zero-initialization verification: PASS")
print(f"wrapped output equals frozen linear output: {matches_base}")

Zero-initialization verification: PASS
wrapped output equals frozen linear output: True


Because `B` starts at zero, the low-rank branch adds exactly zero on the first forward pass. That preserves the pretrained linear transformation at initialization instead of perturbing it before any learning happens.

In [7]:
count_linear = nn.Linear(4096, 4096)
count_lora = LoRALinear(count_linear, rank=16, alpha=16)

trainable_params = sum(parameter.numel() for parameter in count_lora.parameters() if parameter.requires_grad)
total_params = sum(parameter.numel() for parameter in count_lora.parameters())
manual_trainable_params = 2 * 4096 * 16
manual_total_params = (4096 * 4096) + 4096 + manual_trainable_params
trainable_fraction = 100 * trainable_params / total_params

assert trainable_params == manual_trainable_params
assert total_params == manual_total_params

print("Parameter count verification: PASS")
print(f"trainable params: {trainable_params}")
print(f"total params: {total_params}")
print(f"trainable fraction (%): {trainable_fraction:.6f}")
print(f"manual trainable params: {manual_trainable_params}")
print(f"manual total params: {manual_total_params}")

Parameter count verification: PASS
trainable params: 131072
total params: 16912384
trainable fraction (%): 0.775006
manual trainable params: 131072
manual total params: 16912384


## Task 3.3 - Conceptual Questions

### Q1
`B` must start at zero so the LoRA path contributes nothing on the first forward pass. That guarantees the wrapped module initially behaves exactly like the pretrained linear layer, which is the safe starting point for fine-tuning. If both `A` and `B` were random, their product would inject a random low-rank perturbation into every output before training begins. That would immediately distort the pretrained representation space and make optimization start from a noisier, worse state.

### Q2
The `alpha / rank` factor separates adapter capacity from adapter strength. If `alpha = rank`, the scale becomes 1.0, but keeping `alpha` independent lets you increase or decrease the magnitude of the LoRA update without changing how many trainable directions the adapter has. A larger alpha makes the learned low-rank branch influence the output more strongly, while a smaller alpha dampens it. That means rank controls expressiveness, whereas alpha controls how aggressively that expressiveness is applied during training.

### Q3
LoRA is similar to Week 3 because both approaches freeze a large pretrained backbone and only train a much smaller set of parameters on top of it. The fundamental difference is that a classification head only changes the final mapping from features to labels, while LoRA changes the behavior of internal linear transformations throughout the network. A generative language model needs its internal token representations, attention patterns, and MLP activations to shift so it can produce different sequences, not just different labels at the end. LoRA solves that internal adaptation problem by learning a small update inside the model itself, which a new output head alone cannot do for autoregressive text generation.